# agent-guard: run everything on a Colab T4
**Runtime > Change runtime type > T4 GPU** first. Then run cells top to bottom. Each model runs in its own try/except, so one failure does not stop the rest.
Total time roughly 1-2 hours. At the end you download `results.zip` and send it to Claude.

In [ ]:
!nvidia-smi | head -5
import torch; print('cuda:', torch.cuda.is_available())

## 1. Upload the repo
On your Mac run: `cd ~/Desktop && zip -r agent-guard.zip agent-guard -x '*/.git/*'`, then upload that zip here.

In [ ]:
from google.colab import files
up = files.upload()           # choose agent-guard.zip
!unzip -q -o agent-guard.zip -d /content
%cd /content/agent-guard
!pip install -q -r requirements.txt

## 2. Optional: Hugging Face login (needed only for the gated Llama Prompt Guard 2 baseline and for uploading)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # paste a token; skip this cell if you don't have one

## 3. Build data + leakage check

In [ ]:
%cd /content/agent-guard/src
!python build_train.py && python build_redteam.py && python dedupe_check.py
!python tfidf_baseline.py && python tfidf_baseline.py --normalize

## 4. 'Before training': zero-shot baselines

In [ ]:
import subprocess, os, time
os.makedirs('../results/logs', exist_ok=True)
def run(name, cmd):
    t=time.time(); p=subprocess.run(cmd, shell=True, capture_output=True, text=True)
    open(f'../results/logs/{name}.log','w').write(p.stdout+'\n'+p.stderr)
    print(('OK  ' if p.returncode==0 else 'FAIL'), name, f'{time.time()-t:.0f}s')
    if p.returncode: print(p.stderr[-1500:])
run('zs_qwen05b',    'python baselines.py --kind judge --model Qwen/Qwen2.5-0.5B-Instruct --name zs_qwen05b')
run('zs_qwen15b',    'python baselines.py --kind judge --model Qwen/Qwen2.5-1.5B-Instruct --name zs_qwen15b')
run('zs_protectai',  'python baselines.py --kind clf --model protectai/deberta-v3-base-prompt-injection-v2 --name zs_protectai')
run('zs_promptguard','python baselines.py --kind clf --model meta-llama/Llama-Prompt-Guard-2-86M --name zs_promptguard')

## 5. 'After training': fine-tune detectors

In [ ]:
run('modernbert',      'python train.py --model answerdotai/ModernBERT-base --name modernbert')
run('modernbert_norm', 'python train.py --model answerdotai/ModernBERT-base --name modernbert_norm --normalize')
run('deberta',         'python train.py --model microsoft/deberta-v3-base --name deberta')
run('qwen05b_lora',    'python train.py --model Qwen/Qwen2.5-0.5B --name qwen05b_lora --lora')

## 6. Leave-one-technique-out adversarial fine-tune (generalisation to unseen bypass techniques)

In [ ]:
%cd /content/agent-guard/src
!pip uninstall -y -q torchao
s = open('train.py').read()
s = s.replace('warmup_ratio=0.1', 'warmup_steps=max(1, int(0.1 * np.ceil(len(train) / a.bs) * a.epochs))')
open('train.py','w').write(s)
print('patched:', 'warmup_steps' in s)

In [ ]:
run('modernbert',      'python train.py --model answerdotai/ModernBERT-base --name modernbert')
run('modernbert_norm', 'python train.py --model answerdotai/ModernBERT-base --name modernbert_norm --normalize')

In [ ]:
%cd /content/agent-guard/src
s = open('train.py').read()
s = s.replace('model.config.pad_token_id = tok.pad_token_id',
              'model = model.float()\nmodel.config.pad_token_id = tok.pad_token_id', 1)
open('train.py','w').write(s)
print('patched:', 'model.float()' in s)

In [ ]:
run('deberta',      'python train.py --model microsoft/deberta-v3-base --name deberta')
run('qwen05b_lora', 'python train.py --model Qwen/Qwen2.5-0.5B --name qwen05b_lora --lora')

In [ ]:
import json
techs = sorted({json.loads(l)['technique'] for l in open('../data/processed/redteam.jsonl')})
for T in ['b64_pipe','zero_width','nonenglish_injection','indirect_makefile']:
    aug = ','.join(t for t in techs if t != T)
    run(f'loto_{T}', f'python train.py --model answerdotai/ModernBERT-base --name loto_{T} --aug_techniques {aug}')

## 7. Evaluate everything + results table

In [ ]:
%cd /content/agent-guard/src
import glob, os, subprocess
for f in sorted(glob.glob('../results/scores_*.jsonl')):
    print('=== ', os.path.basename(f))
    r = subprocess.run(f'python eval.py --scores {f}', shell=True, capture_output=True, text=True)
    print('\n'.join(r.stdout.splitlines()[:8]) if r.returncode == 0 else 'FAIL: ' + r.stderr[-500:])
r = subprocess.run('python make_table.py', shell=True, capture_output=True, text=True)
print(r.stdout or r.stderr[-800:])

## 8. Package results (send results.zip to Claude)

In [ ]:
import shutil; from google.colab import files
shutil.make_archive('/content/results','zip','/content/agent-guard/results')
files.download('/content/results.zip')

## 9. Optional: publish the best model + dataset to Hugging Face (make them public afterwards)
Edit `USER` first.

In [ ]:
from huggingface_hub import HfApi
USER='shereen20'; api=HfApi()
api.create_repo(f'{USER}/agent-guard-modernbert', exist_ok=True)
api.upload_folder(folder_path='/content/agent-guard/artifacts/modernbert', repo_id=f'{USER}/agent-guard-modernbert')
api.create_repo(f'{USER}/agent-guard-data', repo_type='dataset', exist_ok=True)
api.upload_folder(folder_path='/content/agent-guard/data', repo_id=f'{USER}/agent-guard-data', repo_type='dataset')